# 🧭 VRP Chat — Notebook de Integração com LLM

Este notebook carrega o arquivo `melhor_solucao.json` gerado pelo script VRP do arquivo caixeiro_viajante.py, cria um contexto e disponibiliza um chat interativo (com integração à API OpenAI).

In [ ]:
from IPython.display import display

import ipywidgets as widgets
import json
import re
import sys
import openai

# Adiciona o diretório raiz do projeto ao sys.path sys.path.append('..') # se o notebook está dentro de notebooks/
sys.path.append('..') # se o notebook está dentro de notebooks/

# Carregar melhor solução do JSON
with open("../melhor_solucao.json", "r", encoding="utf-8") as f:
    data = json.load(f)

best_solution = data["melhor_solucao"]
distancia_total = data["distancia_total"]

instrucoes_por_veiculo = {int(k): v for k, v in data["instrucoes_motoristas"].items()}


def ask_llm(query_text):
    # Detecta se é pergunta por veículo específico
    match = re.search(r"ve[ií]culo\s*(\d+)", query_text.lower())
    
    if match:
        vid = int(match.group(1))
        instrucao = instrucoes_por_veiculo.get(vid)
        if instrucao:
            context_text = instrucao
        else:
            context_text = f"Veículo {vid} não encontrado."
    else:
        # Pergunta geral: concatena instruções de todos os veículos
        context_text = "\n\n".join([f"Veículo {vid}: {txt}" for vid, txt in instrucoes_por_veiculo.items()])

    system_prompt = "Você é um assistente logístico. Responda em português e de forma objetiva."
    
    user_prompt = f"Contexto:\n{context_text}\n\nPergunta: {query_text}"
    
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.6,
        max_tokens=1600  # aumenta para não truncar respostas longas
    )
    
    return response.choices[0].message.content


# Widgets de interação
input_box = widgets.Text(placeholder="Digite sua pergunta...")
output_box = widgets.Output()
submit_button = widgets.Button(description="Enviar")

def on_submit(_):
    query = input_box.value.strip()
    if query:
        with output_box:
            print(f"🧑‍💼 Você: {query}")
            resposta = ask_llm(query)
            print(f"🤖 Assistente: {resposta}\n")
        input_box.value = ""  # Limpa a caixa de texto

submit_button.on_click(on_submit)

display(input_box, submit_button, output_box)


Text(value='', placeholder='Digite sua pergunta...')

Button(description='Enviar', style=ButtonStyle())

Output()

## ▶️ Exemplos de perguntas
- Qual a rota do veículo 1?
- Qual veículo está mais carregado?
- Qual a melhor rota?
- Quantas paradas o veículo 2 tem?